In [ ]:
# Cel (22.08.2026): dokoncz test "artefakt P_days~dlugosc cyklu vs realny
# efekt" - polacz istniejace wyniki (P_days=1675/3350, z cycle_scan_*.csv)
# z nowa siatka (P_days=2100..4000, z pgrid_scan_*.csv, 20260822g.ipynb) i
# policz TREND korelacji (pochodna LOWESS vs sila efektu) w funkcji P_days.
# CZYTA WYLACZNIE juz zapisane pliki z results/ - bezpieczny do
# wielokrotnego odpalania (nie wywoluje zadnego skanu).
import glob
import os
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.signal import find_peaks
from statsmodels.nonparametric.smoothers_lowess import lowess
from scipy.stats import spearmanr, pearsonr

RESULTS_DIR = "../results"
STATIONS_5 = ["mosc", "oulu", "athn", "hrms", "sopb"]
FILENAME_RE = re.compile(r"^(?P<prefix>cycle_scan|pgrid_scan)_(?P<station>[a-z]+)_(?P<cycle>cycle_\d{4}(?:_incomplete)?)_P(?P<Pdays>\d+)_d(?P<d>\d+)\.csv$")


def parse_result_filename(path):
    m = FILENAME_RE.match(os.path.basename(path))
    if not m or m["station"] not in STATIONS_5:
        return None
    return dict(station=m["station"], P_days=int(m["Pdays"]), d=int(m["d"]))


frames = []
paths = sorted(glob.glob(os.path.join(RESULTS_DIR, "cycle_scan_*.csv")) +
                glob.glob(os.path.join(RESULTS_DIR, "pgrid_scan_*.csv")))
for path in paths:
    meta = parse_result_filename(path)
    if meta is None:
        continue
    df = pd.read_csv(path, usecols=["t0", "PPDF"], parse_dates=["t0"]).dropna(subset=["PPDF"])
    if len(df) == 0:
        continue
    df["station"] = meta["station"]
    df["P_days"] = meta["P_days"]
    df["d"] = meta["d"]
    frames.append(df)

all_candidates = pd.concat(frames, ignore_index=True)
print(f"Wczytano {len(all_candidates)} wierszy (5 stacji) z {len(frames)} plikow")
print(f"Dostepne P_days: {sorted(all_candidates['P_days'].unique())}")


In [ ]:
# Pochodna LOWESS aktywnosci slonecznej - identycznie jak 20260822f.ipynb.
sn = pd.read_csv("../data/SN_m_tot_V2.0.csv", sep=";")
sn.columns = [c.strip() for c in sn.columns]
sn["date"] = pd.to_datetime(dict(year=sn["rok"], month=sn["miesiac"], day=1))
smoothed = lowess(sn["sunspot"].values, sn["rok_norm"].values, frac=0.01)[:, 1]
sn_derivative = np.gradient(smoothed, sn["rok_norm"].values)


def solar_derivative_at(dates):
    x = mdates.date2num(sn["date"].to_numpy())
    return np.interp(mdates.date2num(pd.DatetimeIndex(dates)), x, sn_derivative)


# Korelacja per stacja/P_days/d na CALEJ siatce (1675..4000).
rows = []
for (station, P_days, d), g in all_candidates.groupby(["station", "P_days", "d"]):
    if len(g) < 30:
        continue
    strength = -np.log10(np.clip(g["PPDF"].to_numpy(dtype=float), 1e-300, None))
    deriv = solar_derivative_at(g["t0"].to_numpy())
    rho, p_spear = spearmanr(deriv, strength)
    rows.append(dict(station=station, P_days=P_days, d=d, n=len(g), spearman_rho=rho))

corr_df = pd.DataFrame(rows).sort_values(["P_days", "d", "station"])
corr_df.to_csv(f"{RESULTS_DIR}/pgrid_solar_derivative_correlation.csv", index=False)
pd.set_option("display.width", 160)
print(corr_df.to_string(index=False))


In [ ]:
# GLOWNY WYKRES: Spearman rho (pochodna vs sila efektu) w funkcji P_days -
# per stacja (cienkie linie) + mediana (gruba czarna), osobno d=1/d=5.
# Pionowa szara linia = srednia dlugosc wykrytego pelnego cyklu w tym repo
# (~10.9 roku = ok. 3987 dni, patrz 20260822c.ipynb komorka 2) - jesli
# korelacja rosnie AZ DO tej linii i dopiero potem plateau/spada, to mocny
# argument za artefaktem "P_days~dlugosc cyklu"; jesli rosnie GLADKO i
# dalej ROSNIE poza nia, bardziej pasuje do realnego efektu; jesli ma
# wyrazne, izolowane maksimum WYRAZNIE PONIZEJ dlugosci cyklu, to jeszcze
# inna historia do zbadania.
AVG_CYCLE_DAYS = 3987

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
for ax, d in zip(axes, [1, 5]):
    sub = corr_df[corr_df["d"] == d]
    for station, g in sub.groupby("station"):
        g = g.sort_values("P_days")
        ax.plot(g["P_days"], g["spearman_rho"], "o-", alpha=0.4, lw=1, label=station)
    median_line = sub.groupby("P_days")["spearman_rho"].median().sort_index()
    ax.plot(median_line.index, median_line.values, "o-", color="black", lw=2.5, label="mediana (5 stacji)")
    ax.axhline(0, color="gray", lw=0.8)
    ax.axvline(AVG_CYCLE_DAYS, color="red", ls=":", lw=1.2, label="śr. długość cyklu (~10.9 lat)")
    ax.set_xlabel("P_days")
    ax.set_title(f"d={d}")
    ax.grid(ls=":", alpha=0.4)
axes[0].set_ylabel("Spearman rho (pochodna LOWESS vs -log10(PPDF))")
axes[1].legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
fig.suptitle("Korelacja siły efektu z pochodną aktywności słonecznej w funkcji P_days\n(5 stacji reprezentatywnych)")
fig.tight_layout()
fig.savefig(f"{RESULTS_DIR}/pgrid_solar_derivative_correlation_trend.png", dpi=120, bbox_inches="tight")
plt.show()
